In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import pickle
import os

In [2]:
RANDOM_SEED = 42
TEST_SIZE = 0.4
VAL_TEST_RATIO = 0.5
OUTPUT_DIR = 'splits'

data = pd.read_csv('../CR_real_masks_more_labeled_veritices_agreed.csv')
data['node_id1'] -= 1
data['node_id2'] -= 1

In [3]:
labels = data['label_id1'].unique().tolist() + data['label_id2'].unique().tolist()
labels = sorted(list(set([l for l in labels if l != 'masked'])))

df = pd.concat([
    data[['node_id1', 'label_id1']].rename(columns={'node_id1': 'node_id', 'label_id1': 'label'}),
    data[['node_id2', 'label_id2']].rename(columns={'node_id2': 'node_id', 'label_id2': 'label'})
], ignore_index=True)
df = df.drop_duplicates(subset=['node_id'])

max_node = max(data['node_id1'].max(), data['node_id2'].max())
node_labels = np.full(max_node + 1, -1, dtype=np.int32)

known_nodes = []
unknown_nodes = []

for _, row in tqdm(df.iterrows(), desc="Processing nodes"):
    if row['label'] == 'masked':
        unknown_nodes.append(row['node_id'])
    else:
        node_labels[row['node_id']] = labels.index(row['label'])
        known_nodes.append(row['node_id'])

known_nodes = sorted(list(set(known_nodes)))
unknown_nodes = sorted(list(set(unknown_nodes)))

Processing nodes: 230887it [00:08, 27018.14it/s]


In [4]:
known_nodes_store = known_nodes.copy()

In [5]:
import random

known_set = set(known_nodes)

random.shuffle(known_nodes)
pairs = []
for i in range(0, len(known_nodes) - 1, 2):
    node1 = known_nodes[i]
    node2 = known_nodes[i + 1]
    coef = random.randint(1, 9) / 10
    pairs.append((node1, node2, coef))

In [6]:
back_labels = {val: i for i, val in enumerate(labels)}
back_labels

{'Belarusians': 0,
 'Northen Russians': 1,
 'Southern Russians': 2,
 'Ukranians': 3}

In [7]:
from collections import defaultdict
from tqdm import tqdm

nodes = np.zeros((max_node + 1 + len(pairs), len(labels)), dtype=np.float32)
edges = defaultdict(list)
for _, row in tqdm(data.iterrows(), total=len(data)):
    if row['label_id1'] != 'masked':
        nodes[row['node_id1'], back_labels[row['label_id1']]] = 1.0
    else:
        nodes[row['node_id1']] = np.ones(len(labels)) / len(labels)
    if row['label_id2'] != 'masked':
        nodes[row['node_id2'], back_labels[row['label_id2']]] = 1.0
    else:
        nodes[row['node_id2']] = np.ones(len(labels)) / len(labels)
    edges[row['node_id1']].append(row['node_id2'])
    edges[row['node_id2']].append(row['node_id1'])

100%|██████████| 6802907/6802907 [06:32<00:00, 17339.23it/s]


In [8]:
unknown_lens = []
known_lens = []
for u in unknown_nodes:
    unknown_lens.append(len(edges[u]))
for u in known_nodes:
    known_lens.append(len(edges[u]))
print(np.mean(unknown_lens), np.mean(known_lens))

29.711174167086547 1485.763436218433


In [9]:
original_known_nodes_count = len(known_nodes)

In [10]:
for i, (node1, node2, coef) in tqdm(enumerate(pairs), total=len(pairs)):
    cur_id = max_node + i
    nodes[cur_id] = coef * nodes[node1] + (1 - coef) * nodes[node2]
    mask_edges1 = np.random.rand(len(edges[node1])) < coef
    mask_edges2 = np.random.rand(len(edges[node2])) < (1 - coef)
    edges[cur_id] = np.concatenate([
        np.array(edges[node1], dtype=np.int32)[mask_edges1],
        np.array(edges[node2], dtype=np.int32)[mask_edges2]
    ]).tolist()
    edges[cur_id] = list(set(edges[cur_id]))
    for u in edges[cur_id]:
        edges[u].append(cur_id)
    known_nodes.append(cur_id)

100%|██████████| 2316/2316 [00:02<00:00, 1087.10it/s]


In [11]:
known_nodes = known_nodes[original_known_nodes_count:]

In [12]:
train_nodes, temp_nodes = train_test_split(
    known_nodes, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
val_nodes, test_nodes = train_test_split(
    temp_nodes, test_size=VAL_TEST_RATIO, random_state=RANDOM_SEED
)

train_nodes = sorted(train_nodes)
val_nodes = sorted(val_nodes)
test_nodes = sorted(test_nodes)

In [13]:
node_labels_masked = nodes.copy()
for n in test_nodes:
    node_labels_masked[n] = np.ones(len(labels)) / len(labels)

In [14]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.save(os.path.join(OUTPUT_DIR, 'train_nodes.npy'), np.array(train_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'val_nodes.npy'), np.array(val_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'test_nodes.npy'), np.array(test_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'unknown_nodes.npy'), np.array(unknown_nodes, dtype=np.int32))
np.save(os.path.join(OUTPUT_DIR, 'node_labels.npy'), nodes)
np.save(os.path.join(OUTPUT_DIR, 'node_labels_masked.npy'), node_labels_masked)

In [15]:
pd_pairs = []
for u, vs in tqdm(edges.items(), total=len(edges)):
    for v in vs:
        pd_pairs.append((u, v))
pd_pairs = pd.DataFrame(pd_pairs, columns=['node_id1', 'node_id2'])

100%|██████████| 233202/233202 [00:03<00:00, 62487.06it/s] 


In [16]:
pd_pairs.to_csv(os.path.join(OUTPUT_DIR, 'edges_data.csv'), index=False)

In [17]:
with open(os.path.join(OUTPUT_DIR, 'labels.txt'), 'w') as f:  # here just label names
    for label in labels:
        f.write(f"{label}\n")

In [18]:
test_ground_truth = {
    'node_ids': np.array(test_nodes, dtype=np.int32),
    'true_labels': np.array([nodes[n] for n in test_nodes], dtype=np.float32),
    'label_names': labels
}
with open(os.path.join(OUTPUT_DIR, 'test_ground_truth.pkl'), 'wb') as f:
    pickle.dump(test_ground_truth, f)
